# Exam: Text Analysis and Visualization

This exam tests your ability to perform text analysis on a dataset of Amazon reviews.
You will be working with the "amazon_review.csv" dataset.
For each question, provide the Python code to perform the requested analysis and generate the visualization.

In [1]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from textblob import TextBlob, Word
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import re

# Download necessary NLTK data
try:
    nltk.download('punkt')
    nltk.download('stopwords')
    nltk.download('averaged_perceptron_tagger')
    nltk.download('wordnet')
except Exception as e:
    print(f"Error downloading NLTK data: {e}")

# Load the dataset
try:
    import os
    filepath = "./datasets/amazon_review.csv"
    if not os.path.exists(filepath):
        filepath = "../datasets/amazon_review.csv"

    df = pd.read_csv(filepath)
    # Drop rows with missing reviews
    df.dropna(subset=['review_text'], inplace=True)
    print(f"Dataset 'amazon_review.csv' loaded successfully from {filepath}.")
except FileNotFoundError:
    print("Error: The dataset file was not found. Please ensure 'amazon_review.csv' is in the 'datasets' directory.")
    print("Attempted paths: './datasets/' and '../datasets/'")
except Exception as e:
    print(f"An error occurred: {e}")


Error: The dataset file was not found. Please ensure 'amazon_review.csv' is in the 'datasets' directory.
Attempted paths: './datasets/' and '../datasets/'


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bapt7\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bapt7\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\bapt7\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\bapt7\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Question 1: Sentiment Analysis and Distribution
1.   Can you perform a sentiment analysis on the collection of product reviews, classify each one as 'positive', 'negative', or 'neutral', and then generate a bar chart to visualize the distribution of these sentiments?"

In [2]:
# Answer 1:
if 'df' in locals():
    sentiments = []
    for review in df['review_text']:
        blob = TextBlob(str(review))
        if blob.sentiment.polarity > 0.1:
            sentiments.append("positive")
        elif blob.sentiment.polarity < -0.1:
            sentiments.append("negative")
        else:
            sentiments.append("neutral")

    df['sentiment'] = sentiments
    sentiment_counts = Counter(sentiments)

    plt.figure(figsize=(8, 6))
    sns.barplot(x=list(sentiment_counts.keys()), y=list(sentiment_counts.values()), palette=['green', 'red', 'blue'])
    plt.title('Sentiment Distribution of Amazon Reviews')
    plt.xlabel('Sentiment')
    plt.ylabel('Number of Reviews')
    plt.show()


## Question 2: Text Preprocessing and Lemmatization
2.   Convert the text to lowercase, remove punctuation and numbers, filter out common stopwords, and then lemmatize the remaining words using their appropriate part-of-speech tags.

In [3]:
# Answer 2:
if 'df' in locals():
    stop_words = set(stopwords.words('english'))

    def get_wordnet_pos(treebank_tag):
        if treebank_tag.startswith('J'): return 'a'
        elif treebank_tag.startswith('V'): return 'v'
        elif treebank_tag.startswith('N'): return 'n'
        elif treebank_tag.startswith('R'): return 'r'
        else: return 'n'

    lemmatized_reviews = []
    for review in df['review_text']:
        blob = TextBlob(str(review).lower())
        lemmatized_words = []
        for word, tag in blob.tags:
            word_cleaned = re.sub(r'[^\w\s]', '', str(word))
            word_cleaned = re.sub(r'\d+', '', word_cleaned).strip()
            if word_cleaned and word_cleaned not in stop_words and len(word_cleaned) > 1:
                wn_tag = get_wordnet_pos(tag)
                lemma = Word(word_cleaned).lemmatize(wn_tag)
                lemmatized_words.append(lemma)
        lemmatized_reviews.append(" ".join(lemmatized_words))

    df['lemmatized_text'] = lemmatized_reviews
    print("Lemmatization and preprocessing complete.")
    print("Sample of lemmatized review:")
    print(df['lemmatized_text'].iloc[0])


## Question 3: Top 15 Most Common Words
3.  After the text has been fully preprocessed and lemmatized, could you calculate the frequency of each unique word and then display a bar graph showing the top 15 most common words?

In [4]:
# Answer 3:
if 'df' in locals() and 'lemmatized_text' in df.columns:
    all_words = " ".join(df['lemmatized_text']).split()
    word_counts = Counter(all_words)
    most_common_words = word_counts.most_common(15)

    words, counts = zip(*most_common_words)

    plt.figure(figsize=(12, 7))
    sns.barplot(x=list(words), y=list(counts))
    plt.title('Top 15 Most Common Lemmatized Words')
    plt.xlabel('Words')
    plt.ylabel('Frequency')
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


## Question 4: Word Cloud
4.  Generate a word cloud based on the entire corpus of lemmatized review words to provide a visual summary of the most prominent terms.

In [5]:
# Answer 4:
if 'df' in locals() and 'lemmatized_text' in df.columns:
    full_lemmatized_text = " ".join(df['lemmatized_text'])
    if full_lemmatized_text.strip():
        wordcloud = WordCloud(width=800, height=400, background_color='white').generate(full_lemmatized_text)
        plt.figure(figsize=(10, 8))
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis('off')
        plt.title('Word Cloud of Amazon Reviews')
        plt.show()
    else:
        print("No text available to generate a word cloud.")


## Question 5: Part-of-Speech (POS) Frequency
5.  Visualize the frequency of nouns, verbs, and adjectives in this text.

In [6]:
# Answer 5:
if 'df' in locals():
    pos_counts = {'Noun': 0, 'Verb': 0, 'Adjective': 0}
    for review in df['review_text']:
        blob = TextBlob(str(review))
        for _, tag in blob.tags:
            if tag.startswith('NN'):
                pos_counts['Noun'] += 1
            elif tag.startswith('VB'):
                pos_counts['Verb'] += 1
            elif tag.startswith('JJ'):
                pos_counts['Adjective'] += 1

    plt.figure(figsize=(8, 6))
    sns.barplot(x=list(pos_counts.keys()), y=list(pos_counts.values()))
    plt.title('Frequency of Nouns, Verbs, and Adjectives')
    plt.xlabel('Part-of-Speech')
    plt.ylabel('Total Count')
    plt.show()
